## Care home identification notebook


this notebook is the latest version as of 17/08/2024 for identifying care home residents in connected bradford after some issues wih previous scripts were identified. it is the number that will be used in the final analysis, having it in a jupytner notebook format also enabls it to be easily upliaded to github to be made freely available.

In [1]:
library(bigrquery)
library(tidyverse) 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
ProjectId = "yhcr-prd-bradfor-bia-core"

targetdb1 <- "yhcr-prd-bradfor-bia-core.CB_2172"
targetdb1 <- gsub(" ","",targetdb1)

targetdb2 <- 'yhcr-prd-bradfor-bia-core.CB_FDM_DeathCertificates'
targetdb2 <- gsub(" ","",targetdb2)

targetdb3 <- "yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare"
targetdb3 <- gsub(" ","",targetdb3)



In [3]:
#first query selects all individuals admitted to a care home at any point no age or date restrictions

In [4]:
sql1 <- 
paste(
    #updated care home cohort code, uses analytic functions so is tidier, this is the currnet query used to define the cohort,
    'with a  as (
SELECT person_id, 
tbl_adultsocialcare_services_start_date, 
servicetype, 
FinalisedPSR, 
row_number() over
(partition by person_id order by tbl_adultsocialcare_services_start_date) as care_seq 
from ',targetdb3,'.tbl_adultsocialcare_services 
where ServiceType in("Nursing","Residential") 
),
b as (
select 
* 
from a
where care_seq = 1
order by person_id, tbl_adultsocialcare_services_start_date),

c as 
(SELECT person_id,
tbl_adultsocialcare_services_end_date, 
servicetype, 
row_number() over (partition by person_id order by tbl_adultsocialcare_services_end_date desc) as care_end_seq 
from ',targetdb3,'.tbl_adultsocialcare_services 
where ServiceType in("Nursing","Residential") )
,

d as (select
* from c 
where care_end_seq = 1),

e as (
select 
b.person_id,
b.tbl_adultsocialcare_services_start_date as first_episodestartdate,
d.tbl_adultsocialcare_services_end_date as last_episodestopdate,
b.ServiceType as admission_service,
b.finalisedPSR
from b 
LEFT join d on
b.person_id= d.person_id
order by person_id),

f as (
Select 
e.*,
datetime_diff(e.last_episodestopdate,e.first_episodestartdate,day) as admission_length
from e
),

h as (
SELECT
f.*,
g.birth_datetime,
FORMAT_DATETIME("%d-%b-%y",first_episodestartdate) as date_of_admission,
g.death_datetime,
g.ethnicity_source_value,
g.gender_source_value
FROM f 
LEFT JOIN ',targetdb3,'.person g on
f.person_id = g.person_id),

g as (
select
DATE_DIFF(first_episodestartdate, birth_datetime, YEAR) as age_admission,
h.*,
from h
where first_episodestartdate < "2019-12-31T00:00:00"
order by age_admission desc)

select
* from g

;',sep="")

sql_tbl <-bq_project_query(ProjectId,sql1)
sql_data <- bq_table_download(sql_tbl)


In [35]:
sql_data

age_admission,person_id,first_episodestartdate,last_episodestopdate,admission_service,finalisedPSR,admission_length,date_of_admission,ethnicity_source_value,gender_source_value,death_date,episodestartdate,episodestopdate,birth_date
<int>,<int>,<dttm>,<dttm>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<date>,<date>,<date>,<date>
110,13165651,2017-10-31,2018-01-09,Residential,Physical Support - Personal Care Support,70,31-Oct-17,Unknown/Refuse to say,F,2018-01-15,2017-10-31,2018-01-09,1907-09-15
108,12773259,2019-04-05,2019-04-15,Nursing,Physical Support - Personal Care Support,10,05-Apr-19,Unknown/Refuse to say,F,2019-04-15,2019-04-05,2019-04-15,1911-07-15
105,12993099,2018-07-20,2018-11-13,Nursing,Physical Support - Personal Care Support,116,20-Jul-18,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2018-11-15,2018-07-20,2018-11-13,1913-03-15
104,12503132,2019-05-07,2019-05-30,Nursing,Physical Support - Personal Care Support,23,07-May-19,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2019-05-15,2019-05-07,2019-05-30,1915-06-15
104,12606511,2019-02-06,2019-05-02,Residential,Physical Support - Personal Care Support,85,06-Feb-19,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2019-05-15,2019-02-06,2019-05-02,1915-02-15
104,13729945,2018-09-03,2019-02-21,Nursing,Physical Support - Personal Care Support,171,03-Sep-18,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2019-02-15,2018-09-03,2019-02-21,1914-10-15
104,12696288,2019-08-23,2021-06-24,Residential,Physical Support - Personal Care Support,671,23-Aug-19,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2021-06-15,2019-08-23,2021-06-24,1915-11-15
104,12757662,2019-10-03,2022-02-15,Residential,Physical Support - Personal Care Support,866,03-Oct-19,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2022-02-15,2019-10-03,2022-02-15,1915-09-15
104,12984164,2019-07-10,2019-07-28,Nursing,Physical Support - Personal Care Support,18,10-Jul-19,White: English or Welsh or Scottish or Northern Irish or British - England and Wales ethnic category 2011 census,F,2019-07-15,2019-07-10,2019-07-28,1915-11-15


In [5]:
#convert vectors to date

In [6]:
sql_data$death_date <- as_date(sql_data$death_datetime)
sql_data <- sql_data %>% select(-death_datetime)

In [7]:
sql_data$episodestartdate <- as_date(sql_data$first_episodestartdate)


In [8]:
sql_data$episodestopdate <- as_date(sql_data$last_episodestopdate)


In [9]:
sql_data$birth_date <- as_date(sql_data$birth_datetime)
sql_data <- sql_data %>% select(-birth_datetime)

In [10]:
# remove those inidividuals admitted on dates that were identified as being erroneous
# (i.e. these individuals were actually admitted into a care home prior to 2016, but had no start date so the date was put as the date of upload, creating some skew) 
# this has been confirmed with the individual who built the social care FDM in Connected bradford

In [11]:
sql_data <- sql_data %>% filter(!(episodestartdate == "2016-04-04"))%>% filter(!(episodestartdate == "2016-07-25"))%>% filter(!(episodestartdate == "2016-08-22"))%>% filter(!(episodestartdate == "2015-11-02"))

In [12]:
sql_data %>% n_distinct('person_id')

[1] 3918

In [13]:
#how many aged less than 65 

In [14]:
sql_data_below_65 <- sql_data %>% filter(age_admission < 65)
n_distinct(sql_data_below_65$person_id)

[1] 333

In [15]:
sql_data_above_sixtyfive <- sql_data %>% filter(age_admission >= 65) 
sql_data_above_sixtyfive %>% n_distinct()


[1] 3585

In [16]:
#here we bring in the mortality data to reduce number of missig death dates 

In [17]:
sql_death <- paste('
with a as (SELECT person_id,
dod,
tbl_NEC_Deaths,
reg_date,
rank() over(partition by person_id order by dod desc, reg_date desc) as rank_seq
from ',targetdb2,'.tbl_NEC_Deaths
where person_id in (SELECT person_id from ',targetdb3,'.person where death_datetime is null)),

b as (
select
*,
row_number() over(partition by person_id order by person_id) as rank_seq_2
from a 
where rank_seq = 1)

select 
person_id, dod,reg_date from b 
where rank_seq_2 = 1 
order by person_id 
;',sep="")

tbl_death <-bq_project_query(ProjectId,sql_death)
ch_death<- bq_table_download(tbl_death)



In [18]:
ch_death <- ch_death %>% mutate(date_death = ifelse(is.na(dod), reg_date,dod))

In [19]:
ch_death$date_death <- as_date(ch_death$date_death)

In [20]:
## join death certificate tibble to the deathdate in the master cohort table 

In [21]:
ch_death_filter <- ch_death %>% select(person_id,date_death)
sql_data_above_sixtyfive <- left_join(sql_data_above_sixtyfive,ch_death_filter,by='person_id')

In [22]:
sql_data_above_sixtyfive <- sql_data_above_sixtyfive %>% mutate(dod = coalesce(sql_data_above_sixtyfive$death_date,sql_data_above_sixtyfive$date_death))%>% select(-death_date,-date_death)


In [23]:
sql_data_above_sixtyfive <- sql_data_above_sixtyfive %>% mutate(mortality = dod-episodestartdate)

In [24]:
alive_morethan_6weeks <- sql_data_above_sixtyfive %>% filter((mortality > 42) |is.na(dod))

In [25]:
alive_morethan_6weeks %>% n_distinct()

[1] 2945

In [26]:
##next we remove anyone who was discharged within 42 days or died within 42 days
#first we calculate the number and then remove anyone who died within 42 days 
long_stay_resident <- alive_morethan_6weeks %>% filter(episodestopdate > (episodestartdate + ddays(x=42)))


In [27]:
#second, of  those remaining, we calculate and remove those who had a recorded legnth of stay of 42 days or less. 
#the two figures combined (number died withi 42 days and number discharged alive) gives the total number filtered at this stage. 

In [28]:
long_stay_resident %>% n_distinct()

[1] 2884

In [29]:
# we iidentified anyone with missing primary care data and removed them, 
# this is most likely because whilst they are recieving social care funding from the bradford local authorty they are not registered to a GP contributing to Connected Bradford. 

In [30]:
sql_missing <- paste('
select
distinct b.person_id as primary_care_person
from `yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare.person` b
left join (select person_id,
  from `yhcr-prd-bradfor-bia-core.CB_FDM_PrimaryCare.tbl_srcode`
  where person_id in(select person_id from `yhcr-prd-bradfor-bia-core.CB_FDM_AdultSocialCare.person`)) a
  on b.person_id = a.person_id
  where a.person_id is null

                     ;',sep="")

tbl_missing <-bq_project_query(ProjectId,sql_missing)
tbl_missing <- bq_table_download(tbl_missing)


Warning message in as.integer.integer64(x):
“NAs produced by integer overflow”


In [31]:
care_home_cohort_final <- long_stay_resident %>% filter(!(long_stay_resident$person_id %in% tbl_missing$primary_care_person)) 
care_home_cohort_final %>%  n_distinct()
care_home_ghosts <- care_home_cohort_final %>% filter( is.na(dod))

[1] 2793

the 'ghosts' are those who had a missing death date 

In [32]:
care_home_ghosts %>% n_distinct()

[1] 147

In [33]:
#send the final care home cohort back to bigquery as a table so it can be ran through the other queries
care_home_cohort_final$mortality <- as.numeric(care_home_cohort_final$mortality)

In [34]:
mybq = bq_table(project='yhcr-prd-bradfor-bia-core', dataset='CB_2172', table='care_home_cohort_v1')
bq_table_upload(x=mybq, values= care_home_cohort_final, create_disposition='CREATE_IF_NEEDED', 
             write_disposition='WRITE_TRUNCATE')

ERROR: [1m[33mError[39m in `bq_table_upload()`:[22m
[1m[22m[33m![39m Job
  yhcr-prd-bradfor-bia-core.job_AUwhSMj6O0p3diWmp0qKMIOTc2Zc.europe-west2
  failed
[31m✖[39m Error while reading data, error message: JSON processing encountered too many
  errors, giving up. Rows: 1; errors: 1; max bad: 0; error percent: 0 [invalid]
[31m✖[39m Invalid date: '31-Oct-17' Field: date_of_admission; Value: 31-Oct-17
  [invalidQuery]


In [ ]:
sql_ch <- paste('
with a as (
select
* except(first_episodestartdate,finalisedPSR,episodestartdate,episodestopdate), 
cast(first_episodestartdate as DATETIME) as first_episodestartdate
from ',targetdb1,'.care_home_cohort_v1)

select
a.*,
b.observation_period_end_date
from a
left join `yhcr-prd-bradfor-bia-core.CB_FDM_PrimaryCare.observation_period` b
on a.person_id = b.person_id

                     ;',sep="")

tbl_ch <-bq_project_query(ProjectId,sql_ch)
tbl_ch <- bq_table_download(tbl_ch)



this section merges the data from 

In [ ]:
tbl_ch_dod_observation_merged <- tbl_ch %>% mutate(dod = coalesce(tbl_ch$dod,tbl_ch$observation_period_end_date))
